In [ ]:
from huggingface_hub import login, whoami
whoami()

!pip install wandb -Uq

import os, wandb

# ── dedicated workspace for final runs ────────────────────────────────
WANDB_PROJECT = "mdlm-sft-final"            # new project = fresh workspace
WANDB_GROUP   = "stage2-reasoning"           # optional: groups related runs in the UI

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


False

In [ ]:
import tempfile
from pathlib import Path
from huggingface_hub import list_bucket_tree, download_bucket_files


BUCKET = "avgJo3/final-mdlm-sft-artifacts"

PREFIXES: dict[str, str] = {
    "base":      "20260714T050657-0e2e053501df5170/model_",
    "reasoning": "20260715T045644-7d0f95338c3101b3/model_",
}

CHAIN_STEMS = {
    "train":        "train",
    "mix":          "mix",
    "ablation":     "ablation",
    "mix-ablation": "mix-ablation",
}
ROUNDS = ["R0", "R1", "R2", "R3", "R4"]

_MODEL_TEMPDIRS: dict[str, tempfile.TemporaryDirectory] = {}


def _round_dirname(stem: str, k: int) -> str:
    """
    Directory naming convention:
      - k == 0 : always `model_trained_r0` (shared R0, regardless of stem).
      - k >= 1, stem == 'train'    : `model_trained_r{k}`
      - k >= 1, other stems        : `model_{stem}-trained_r{k}`
    """
    if k == 0:
        return "model_trained_r0"
    if stem == "train":
        return f"model_trained_r{k}"
    return f"model_{stem}-trained_r{k}"


def load_model_paths_for(ckpt_alias: str) -> dict[str, dict[str, Path]]:
    if ckpt_alias not in PREFIXES:
        raise KeyError(f"unknown checkpoint alias {ckpt_alias!r}; "
                       f"known aliases: {list(PREFIXES)}")
    prefix = PREFIXES[ckpt_alias]

    files = [
        item for item in list_bucket_tree(BUCKET, prefix=prefix, recursive=True)
        if item.type == "file"
    ]
    if not files:
        raise RuntimeError(f"no files found under prefix: {prefix}")

    tmp = tempfile.TemporaryDirectory(prefix=f"models-{ckpt_alias}-")
    _MODEL_TEMPDIRS[ckpt_alias] = tmp
    tmp_root = Path(tmp.name)

    downloads = [(f, str(tmp_root / f.path)) for f in files]
    download_bucket_files(BUCKET, files=downloads)

    data_root = (tmp_root / prefix).parent

    def resolve_chain(stem: str) -> dict[str, Path]:
        paths = {}
        for k in range(len(ROUNDS)):
            model_dir = data_root / _round_dirname(stem, k)
            assert model_dir.exists(), f"model dir not found: {model_dir}"
            paths[f"R{k}"] = model_dir
        return paths

    return {cid: resolve_chain(stem) for cid, stem in CHAIN_STEMS.items()}


model_paths_by_ckpt: dict[str, dict[str, dict[str, Path]]] = {}
model_paths_by_ckpt["base"]      = load_model_paths_for("base")
model_paths_by_ckpt["reasoning"] = load_model_paths_for("reasoning")

In [ ]:
model_paths_by_ckpt

{'base': {'train': {'R0': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_trained_r0'),
   'R1': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_trained_r1'),
   'R2': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_trained_r2'),
   'R3': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_trained_r3'),
   'R4': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_trained_r4')},
  'mix': {'R0': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_trained_r0'),
   'R1': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_mix-trained_r1'),
   'R2': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_mix-trained_r2'),
   'R3': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_mix-trained_r3'),
   'R4': PosixPath('/tmp/models-base-uog8b_xl/20260714T050657-0e2e053501

In [ ]:
from datasets import load_dataset

dd = load_dataset("avgJo3/writingprompts-strat", split='test')
dd = dd.save_to_disk("writingprompts-strat")

README.md:   0%|          | 0.00/740 [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.47M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.7M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
!git clone https://github.com/AvgJoe-cpu/mdlm_sft.git
%cd mdlm_sft
!pip install -e . -q

Cloning into 'mdlm_sft'...
remote: Enumerating objects: 985, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 985 (delta 46), reused 65 (delta 31), pack-reused 901 (from 1)
Receiving objects: 100% (985/985), 529.09 KiB | 12.90 MiB/s, done.
Resolving deltas: 100% (533/533), done.
/content/mdlm_sft
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 169.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 8

In [ ]:
import sys
sys.path.insert(0, "/content/mdlm_sft/src")

from mdlm_sft.mdlm.mdlm_gen_v3 import MDLMGenerationConfig, generate_mdlm, run_inference

In [ ]:
import json
from pathlib import Path
from datasets import DatasetDict, load_from_disk


# --- what to run ------------------------------------------------------------
INPUT_DATASET_PATH = "/content/writingprompts-strat/test"
OUTPUT_ROOT        = Path("./generations")        # persistent scratch

GEN_KNOBS = {
    "response_length": 128,
    "num_steps":       128,
    "batch_size":      64,
}


def _write_dataset_dict_marker(chain_dir: Path, round_keys: list[str]) -> None:
    marker = chain_dir / "dataset_dict.json"
    marker.write_text(json.dumps({"splits": round_keys}))


def run_all_chains(
    model_paths_by_ckpt: dict[str, dict[str, dict[str, Path]]],
    input_path: Path,
    output_root: Path,
    knobs: dict,
    skip_existing: bool = True,
) -> dict[str, dict[str, DatasetDict]]:
    out: dict[str, dict[str, DatasetDict]] = {}

    for ckpt_alias, chains in model_paths_by_ckpt.items():
        out[ckpt_alias] = {}

        for chain_id, rounds in chains.items():
            chain_dir = output_root / ckpt_alias / chain_id

            if skip_existing and (chain_dir / "dataset_dict.json").exists():
                print(f"[skip] {ckpt_alias}/{chain_id} — DatasetDict already saved")
                out[ckpt_alias][chain_id] = DatasetDict.load_from_disk(str(chain_dir))
                continue

            print(f"[chain] {ckpt_alias}/{chain_id}")

            for round_key, model_path in rounds.items():
                round_dir = chain_dir / round_key

                if skip_existing and round_dir.exists() and any(round_dir.iterdir()):
                    print(f"  [skip]  {round_key} — output exists")
                    continue

                print(f"  [round] {round_key}   model: {model_path}")
                round_dir.parent.mkdir(parents=True, exist_ok=True)

                cfg = MDLMGenerationConfig(
                    model_name_or_path  = str(model_path),
                    dataset_input_path  = str(input_path),
                    dataset_output_path = str(round_dir),
                    **knobs,
                )
                run_inference(cfg)

            # promote the directory of loose splits into a DatasetDict
            _write_dataset_dict_marker(chain_dir, list(rounds.keys()))
            print(f"  [save]  DatasetDict marker written at {chain_dir}")

            out[ckpt_alias][chain_id] = DatasetDict.load_from_disk(str(chain_dir))

    return out


# --- entry ------------------------------------------------------------------
generated_chains = run_all_chains(
    model_paths_by_ckpt,
    input_path  = INPUT_DATASET_PATH,
    output_root = OUTPUT_ROOT,
    knobs       = GEN_KNOBS,
)

# `generated_chains` mirrors chains_by_ckpt in structure. Feed downstream:
#
#     for chain_id, dd in generated_chains["base"].items():
#         df, U_hats = process_chain(dd)

[chain] base/train
  [round] R0   model: /tmp/models-base-uog8b_xl/20260714T050657-0e2e053501df5170/model_trained_r0


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model compiled with torch.compile


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is d

...

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model compiled with torch.compile


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

  [save]  DatasetDict marker written at generations/reasoning/mix-ablation


In [ ]:
for chain_id, dd in generated_chains.items():
  print(dd)

{'train': DatasetDict({
    R0: Dataset({
        features: ['prompt', 'completion', 'label', 'id', 'prompt_token_count', 'completion_token_count', 'text_token_count'],
        num_rows: 1000
    })
    R1: Dataset({
        features: ['prompt', 'completion', 'label', 'id', 'prompt_token_count', 'completion_token_count', 'text_token_count'],
        num_rows: 1000
    })
    R2: Dataset({
        features: ['prompt', 'completion', 'label', 'id', 'prompt_token_count', 'completion_token_count', 'text_token_count'],
        num_rows: 1000
    })
    R3: Dataset({
        features: ['prompt', 'completion', 'label', 'id', 'prompt_token_count', 'completion_token_count', 'text_token_count'],
        num_rows: 1000
    })
    R4: Dataset({
        features: ['prompt', 'completion', 'label', 'id', 'prompt_token_count', 'completion_token_count', 'text_token_count'],
        num_rows: 1000
    })
}), 'mix': DatasetDict({
    R0: Dataset({
        features: ['prompt', 'completion', 'label', 'id', 

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from huggingface_hub import create_bucket, list_bucket_tree, sync_bucket

BUCKET = "avgJo3/final-mdlm-sft-artifacts"
SAFESPOT_PREFIX = "HE-ood-gens"


def push_generated_to_bucket(
    output_root: str | Path,
    bucket: str = BUCKET,
    prefix: str = SAFESPOT_PREFIX,
    timestamped: bool = True,
) -> str:
    output_root = Path(output_root)
    assert output_root.exists() and output_root.is_dir(), (
        f"output_root does not exist or is not a directory: {output_root}"
    )

    if timestamped:
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        remote_prefix = f"{prefix}/{stamp}"
    else:
        remote_prefix = prefix

    try:
        create_bucket(bucket)
        print(f"[bucket] created {bucket}")
    except Exception:
        pass

    dest_uri = f"hf://buckets/{bucket}/{remote_prefix}"

    print(f"[sync] {output_root}  →  {dest_uri}")
    plan = sync_bucket(
        source=str(output_root),   # local → bucket direction
        dest=dest_uri,
        verbose=True,
    )
    print(f"[sync] plan summary: {plan.summary()}")

    landed = [
        item for item in list_bucket_tree(bucket, prefix=remote_prefix, recursive=True)
        if item.type == "file"
    ]
    print(f"[sync] {len(landed)} files uploaded under {bucket}/{remote_prefix}")

    return remote_prefix



# safe-spot: push to bucket immediately after generation, before any downstream work
remote_prefix = push_generated_to_bucket(OUTPUT_ROOT)
print(f"[safe-spot] generations available at: {BUCKET}/{remote_prefix}")

# downstream analysis continues on the in-memory dict:
#     for chain_id, dd in generated_chains["base"].items():
#         df, U_hats = process_chain(dd)

[sync] generations  →  hf://buckets/avgJo3/final-mdlm-sft-artifacts/HE-ood-gens/20260715T142404
Sync plan: generations -> hf://buckets/avgJo3/final-mdlm-sft-artifacts/HE-ood-gens/20260715T142404
  Uploads: 128
  Downloads: 0
  Deletes: 0
  Skips: 0
Syncing...
  Uploading: base/ablation/R0/data-00000-of-00001.arrow (new file)
  Uploading: base/ablation/R0/dataset_info.json (new file)
  Uploading: base/ablation/R0/state.json (new file)
  Uploading: base/ablation/R1/data-00000-of-00001.arrow (new file)
  Uploading: base/ablation/R1/dataset_info.json (new file)
  Uploading: base/ablation/R1/state.json (new file)
  Uploading: base/ablation/R2/data-00000-of-00001.arrow (new file)
  Uploading: base/ablation/R2/dataset_info.json (new file)
  Uploading: base/ablation/R2/state.json (new file)
  Uploading: base/ablation/R3/data-00000-of-00001.arrow (new file)
  Uploading: base/ablation/R3/dataset_info.json (new file)
  Uploading: base/ablation/R3/state.json (new file)
  Uploading: base/ablation/R

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ng/ablation/R2/state.json:   2%|1         |  4.00B /   249B            

  ...data-00000-of-00001.arrow:   2%|1         | 14.0kB /  818kB            

...

  ...oning/train/R4/state.json:   2%|1         |  4.00B /   249B            

  ...tion/R0/dataset_info.json:   2%|1         |  33.0B / 1.96kB            

  .../base/train/R4/state.json:   2%|1         |  4.00B /   249B            

  ...ix-ablation/R3/state.json:   2%|1         |  4.00B /   249B            

Sync completed.
[sync] plan summary: {'uploads': 128, 'downloads': 0, 'deletes': 0, 'skips': 0, 'total_size': 32109824}
[sync] 128 files uploaded under avgJo3/final-mdlm-sft-artifacts/HE-ood-gens/20260715T142404
[safe-spot] generations available at: avgJo3/final-mdlm-sft-artifacts/HE-ood-gens/20260715T142404


In [ ]:
!pip install evaluate bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.4 MB/s eta 0:00:00


In [ ]:
from datasets import Dataset, load_dataset

In [ ]:
!pip install mauve-text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 98.7 MB/s eta 0:00:00


In [ ]:
dummy_list = [
    {"idx": 0, "completion": "This is a dummy completion."},
    {"idx": 1, "completion": "Another dummy completion."},
    {"idx": 2, "completion": "Yet another dummy completion."}
]

dummy_compl = [
    {"idx": 0, "completion": "This is a dummy completion."},
    {"idx": 1, "completion": "Another dummy completion."},
    {"idx": 2, "completion": "Yet another dummy completion"}
]

ds = Dataset.from_list(dummy_list)
ds_compl = Dataset.from_list(dummy_compl)
print(ds)

Dataset({
    features: ['idx', 'completion'],
    num_rows: 3
})


In [ ]:
mauve = evaluate.load("mauve")

# dummy data (reusing ds and ds_compl from before)
pred_texts = ds["completion"]
ref_texts  = ds_compl["completion"]

mauve_result = mauve.compute(
    predictions=pred_texts,
    references=ref_texts,
)

print(mauve_result.mauve)        # scalar — the main score
print(mauve_result)              # full output: mauve, frontier_integral, divergence_curve

Loading tokenizer


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Featurizing tokens


Featurizing p:   0%|          | 0/3 [00:00<?, ?it/s]

Tokenizing text...
Featurizing tokens


Featurizing q:   0%|          | 0/3 [00:00<?, ?it/s]

seed = 25
performing clustering in lower dimension = 1
kmeans time: 0.01 s
total discretization time: 0.13 seconds
0.5182012478497631
namespace(p_hist=array([0., 1.]), q_hist=array([0.33333333, 0.66666667]), divergence_curve=array([[1.00000000e+00, 0.00000000e+00],
       [1.00000000e+00, 1.31687572e-01],
       [9.97800591e-01, 1.45988583e-01],
       [9.91094066e-01, 1.61505906e-01],
       [9.79737598e-01, 1.78315528e-01],
       [9.63617793e-01, 1.96496538e-01],
       [9.42654522e-01, 2.16131187e-01],
       [9.16805098e-01, 2.37304951e-01],
       [8.86068867e-01, 2.60106593e-01],
       [8.50492285e-01, 2.84628224e-01],
       [8.10174593e-01, 3.10965368e-01],
       [7.65274197e-01, 3.39217021e-01],
       [7.16015960e-01, 3.69485713e-01],
       [6.62699613e-01, 4.01877572e-01],
       [6.05709635e-01, 4.36502386e-01],
       [5.45527063e-01, 4.73473662e-01],
       [4.82743941e-01, 5.12908691e-01],
       [4.18081458e-01, 5.54928610e-01],
       [3.52413502e-01, 5.99658463e-0

In [ ]:
ds = ds.sort("idx")
ds_compl = ds_compl.sort("idx")
assert ds.features["idx"] == ds_compl.features["idx"]

compl_ref  = ds_compl["completion"]
compl_pred = ds["completion"]

results = []
for i in range(len(compl_ref)):
    result = bertscore.compute(
        predictions=[compl_pred[i]],   # ← list, not string
        references=[compl_ref[i]],     # ← list, not string
        lang="en"
    )
    results.append(result)

print(results)

[{'precision': [1.0], 'recall': [1.0], 'f1': [1.0], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.10.1)'}, {'precision': [1.0000001192092896], 'recall': [1.0000001192092896], 'f1': [1.0000001192092896], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.10.1)'}, {'precision': [0.9908252358436584], 'recall': [0.9885398149490356], 'f1': [0.9896811842918396], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.10.1)'}]


In [ ]:
ds_compl = load_dataset("avgJo3/writingprompts-strat", split="test")
print(ds_compl)

Dataset({
    features: ['prompt', 'completion', 'label', 'id', 'prompt_token_count', 'completion_token_count', 'text_token_count'],
    num_rows: 1000
})


In [ ]:
# compl_ref is fixed across all chains
compl_ref = ds_compl.sort("id")["completion"]

chain_results: dict[str, dict] = {}

for chain_id, dd in generated_chains["base"].items():
    # dd is a DatasetDict with splits R0..R4
    # each split is a Dataset with "idx" and "completion" columns

    round_results: dict[str, list] = {}

    for round_key, round_ds in dd.items():
        round_ds_sorted = round_ds.sort("id")

        # sanity check: same number of examples as reference
        assert len(round_ds_sorted) == len(compl_ref), (
            f"Length mismatch at {chain_id}/{round_key}: "
            f"{len(round_ds_sorted)} vs {len(compl_ref)}"
        )

        compl_pred = round_ds_sorted["completion"]

        result = bertscore.compute(
            predictions=compl_pred,   # full list — one batched call
            references=compl_ref,
            lang="en",
        )
        round_results[round_key] = result   # keys: precision, recall, f1 (lists)
        print(f"  [{chain_id}/{round_key}]  mean F1: {sum(result['f1'])/len(result['f1']):.4f}")

    chain_results[chain_id] = round_results

# chain_results["ablation"]["R0"]["f1"]  → list of per-sample F1 scores

  [train/R0]  mean F1: 0.8000
  [train/R1]  mean F1: 0.7988
  [train/R2]  mean F1: 0.7973
  [train/R3]  mean F1: 0.7964
  [train/R4]  mean F1: 0.7952
  [mix/R0]  mean F1: 0.8001
  [mix/R1]  mean F1: 0.7995
  [mix/R2]  mean F1: 0.7995
  [mix/R3]  mean F1: 0.7992
  [mix/R4]  mean F1: 0.7955
  [ablation/R0]  mean F1: 0.7996
  [ablation/R1]  mean F1: 0.7989
  [ablation/R2]  mean F1: 0.7980
  [ablation/R3]  mean F1: 0.7968
  [ablation/R4]  mean F1: 0.7951
  [mix-ablation/R0]  mean F1: 0.7998
  [mix-ablation/R1]  mean F1: 0.7999
  [mix-ablation/R2]  mean F1: 0.8004
  [mix-ablation/R3]  mean F1: 0.8006
  [mix-ablation/R4]  mean F1: 0.8004


In [ ]:
def evaluate_ckpt_bertscore(
    generated_chains: dict,
    ckpt_alias: str,
    compl_ref: list[str],
    bertscore,
    sort_key: str = "id",
) -> dict[str, dict]:

    chain_results: dict[str, dict] = {}

    for chain_id, dd in generated_chains[ckpt_alias].items():
        round_results: dict[str, dict] = {}

        for round_key, round_ds in dd.items():
            round_ds_sorted = round_ds.sort(sort_key)

            assert len(round_ds_sorted) == len(compl_ref), (
                f"Length mismatch at {chain_id}/{round_key}: "
                f"{len(round_ds_sorted)} vs {len(compl_ref)}"
            )

            compl_pred = round_ds_sorted["completion"]

            result = bertscore.compute(
                predictions=compl_pred,
                references=compl_ref,
                lang="en",
            )
            round_results[round_key] = result
            print(f"  [{ckpt_alias}/{chain_id}/{round_key}]  mean F1: {sum(result['f1'])/len(result['f1']):.4f}")

        chain_results[chain_id] = round_results

    return chain_results

base_results      = evaluate_ckpt_bertscore(generated_chains, "base",      compl_ref, bertscore)
reasoning_results = evaluate_ckpt_bertscore(generated_chains, "reasoning", compl_ref, bertscore)

# access
base_results["mix"]["R0"]["f1"]
reasoning_results["ablation"]["R3"]["precision"]

  [base/train/R0]  mean F1: 0.8000
  [base/train/R1]  mean F1: 0.7988
  [base/train/R2]  mean F1: 0.7973
  [base/train/R3]  mean F1: 0.7964
  [base/train/R4]  mean F1: 0.7952
  [base/mix/R0]  mean F1: 0.8001
  [base/mix/R1]  mean F1: 0.7995
  [base/mix/R2]  mean F1: 0.7995
  [base/mix/R3]  mean F1: 0.7992
  [base/mix/R4]  mean F1: 0.7955
  [base/ablation/R0]  mean F1: 0.7996
  [base/ablation/R1]  mean F1: 0.7989
  [base/ablation/R2]  mean F1: 0.7980
  [base/ablation/R3]  mean F1: 0.7968
  [base/ablation/R4]  mean F1: 0.7951
  [base/mix-ablation/R0]  mean F1: 0.7998
  [base/mix-ablation/R1]  mean F1: 0.7999
  [base/mix-ablation/R2]  mean F1: 0.8004
  [base/mix-ablation/R3]  mean F1: 0.8006
  [base/mix-ablation/R4]  mean F1: 0.8004
  [reasoning/train/R0]  mean F1: 0.7993
  [reasoning/train/R1]  mean F1: 0.7979
  [reasoning/train/R2]  mean F1: 0.7967
  [reasoning/train/R3]  mean F1: 0.7950
  [reasoning/train/R4]  mean F1: 0.7936
  [reasoning/mix/R0]  mean F1: 0.7991
  [reasoning/mix/R1]  

[0.8081012964248657,
 0.8255072832107544,
 0.8179585337638855,
 0.7790026664733887,
 0.7985556721687317,
 0.8032994270324707,
 0.8172561526298523,
 0.8030966520309448,
 0.8169504404067993,
 0.8147944808006287,
 0.8253835439682007,
 0.7864409685134888,
 0.8093770742416382,
 0.8183146119117737,
 0.820064127445221,
 0.8021695613861084,
 0.7838183641433716,
 0.8184525966644287,
 0.8217955827713013,
 0.8230867385864258,
 0.8164619207382202,
 0.8006200194358826,
 0.7998255491256714,
 0.8203233480453491,
 0.7903388738632202,
 0.8092101216316223,
 0.816338837146759,
 0.8126153945922852,
 0.7864147424697876,
 0.810086190700531,
 0.8144674301147461,
 0.8319502472877502,
 0.7890915274620056,
 0.804560661315918,
 0.8129490613937378,
 0.8199700713157654,
 0.8008921146392822,
 0.8138882517814636,
 0.820642352104187,
 0.8308029174804688,
 0.7971081733703613,
 0.7958083152770996,
 0.808753252029419,
 0.8168795108795166,
 0.8161319494247437,
 0.8159641027450562,
 0.8086906671524048,
 0.8037787079811096

In [ ]:
import pandas as pd

rows = []
for ckpt_alias, chain_results in [("base", base_results), ("reasoning", reasoning_results)]:
    for chain_id, round_results in chain_results.items():
        for round_key, result in round_results.items():
            for i, (p, r, f) in enumerate(zip(result["precision"], result["recall"], result["f1"])):
                rows.append({
                    "ckpt_alias": ckpt_alias,
                    "chain_id":   chain_id,
                    "round":      round_key,
                    "idx":        i,
                    "precision":  p,
                    "recall":     r,
                    "f1":         f,
                })

df_bertscore = pd.DataFrame(rows)
print(df_bertscore.head())

  ckpt_alias chain_id round  idx  precision    recall        f1
0       base    train    R0    0   0.820804  0.781634  0.800740
1       base    train    R0    1   0.824797  0.782224  0.802946
2       base    train    R0    2   0.823471  0.777716  0.799940
3       base    train    R0    3   0.812006  0.797774  0.804827
4       base    train    R0    4   0.819216  0.784736  0.801605


In [ ]:
df_summary = (
    df_bertscore
    .groupby(["ckpt_alias", "chain_id", "round"], as_index=False)
    [["precision", "recall", "f1"]]
    .mean()
)
print(df_summary)

   ckpt_alias      chain_id round  precision    recall        f1
0        base      ablation    R0   0.812230  0.787526  0.799586
1        base      ablation    R1   0.811444  0.786915  0.798896
2        base      ablation    R2   0.810413  0.786209  0.798038
3        base      ablation    R3   0.808729  0.785368  0.796791
4        base      ablation    R4   0.806660  0.784077  0.795111
5        base           mix    R0   0.812658  0.788093  0.800099
6        base           mix    R1   0.812157  0.787363  0.799469
7        base           mix    R2   0.811812  0.787706  0.799461
8        base           mix    R3   0.811712  0.787308  0.799226
9        base           mix    R4   0.806059  0.785378  0.795454
10       base  mix-ablation    R0   0.812384  0.787755  0.799761
11       base  mix-ablation    R1   0.812430  0.787983  0.799923
12       base  mix-ablation    R2   0.813224  0.788233  0.800432
13       base  mix-ablation    R3   0.813239  0.788537  0.800591
14       base  mix-ablati

In [ ]:

import evaluate

# ── 1. Load the BERTScore metric ──────────────────────────────────────────────
bertscore = evaluate.load("bertscore")

# ── 2. Dummy dataset (predictions vs. references) ────────────────────────────
predictions = [
    "The cat sat on the mat.",
    "She enjoys reading books in the evening.",
    "The stock market fell sharply today.",
]

references = [
    "A cat was sitting on a mat.",
    "She likes to read books at night.",
    "Stock prices dropped significantly today.",
]

# ── 3. Compute BERTScore ──────────────────────────────────────────────────────
results = bertscore.compute(predictions=predictions, references=references, lang="en")

# ── 4. Display per-sample results ─────────────────────────────────────────────
print(f"{'#':<5} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 45)
for i, (p, r, f) in enumerate(zip(results["precision"], results["recall"], results["f1"])):
    print(f"{i:<5} {p:<12.4f} {r:<12.4f} {f:<12.4f}")

# ── 5. Aggregate (mean) ───────────────────────────────────────────────────────
n = len(predictions)
print("\nMean scores:")
print(f"  Precision : {sum(results['precision']) / n:.4f}")
print(f"  Recall    : {sum(results['recall'])    / n:.4f}")
print(f"  F1        : {sum(results['f1'])        / n:.4f}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#     Precision    Recall       F1          
---------------------------------------------
0     0.9692       0.9651       0.9671      
1     0.9706       0.9633       0.9669      
2     0.9609       0.9712       0.9660      

Mean scores:
  Precision : 0.9669
  Recall    : 0.9665
  F1        : 0.9667


In [ ]:
def evaluate_ckpt_mauve(
    generated_chains: dict,
    ckpt_alias: str,
    compl_ref: list[str],
    mauve,
    sort_key: str = "id",
) -> dict[str, dict]:

    chain_results: dict[str, dict] = {}

    for chain_id, dd in generated_chains[ckpt_alias].items():
        round_results: dict[str, dict] = {}

        for round_key, round_ds in dd.items():
            round_ds_sorted = round_ds.sort(sort_key)

            assert len(round_ds_sorted) == len(compl_ref), (
                f"Length mismatch at {chain_id}/{round_key}: "
                f"{len(round_ds_sorted)} vs {len(compl_ref)}"
            )

            compl_pred = round_ds_sorted["completion"]

            result = mauve.compute(
                predictions=compl_pred,
                references=compl_ref,
            )
            round_results[round_key] = result.mauve   # scalar
            print(f"  [{ckpt_alias}/{chain_id}/{round_key}]  MAUVE: {result.mauve:.4f}")

        chain_results[chain_id] = round_results

    return chain_results


mauve = evaluate.load("mauve")

base_mauve_results      = evaluate_ckpt_mauve(generated_chains, "base",      compl_ref, mauve)
reasoning_mauve_results = evaluate_ckpt_mauve(generated_chains, "reasoning", compl_ref, mauve)

# access
base_mauve_results["mix"]["R0"]           # scalar
reasoning_mauve_results["ablation"]["R3"] # scalar

Tokenizing text...
Featurizing tokens


Featurizing p:   0%|          | 0/1000 [00:00<?, ?it/s]

Tokenizing text...
Featurizing tokens


Featurizing q:   0%|          | 0/1000 [00:00<?, ?it/s]

seed = 25
performing clustering in lower dimension = 402
kmeans time: 0.16 s
total discretization time: 0.51 seconds
  [base/train/R0]  MAUVE: 0.0215
Tokenizing text...
Featurizing tokens


...

seed = 25
performing clustering in lower dimension = 394
kmeans time: 0.17 s
total discretization time: 0.45 seconds
  [reasoning/mix-ablation/R3]  MAUVE: 0.0167
Tokenizing text...
Featurizing tokens


Featurizing p:   0%|          | 0/1000 [00:00<?, ?it/s]

Tokenizing text...
Featurizing tokens


Featurizing q:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
rouge = evaluate.load("rouge")

# dummy data (reusing ds and ds_compl from before)
pred_texts = ds["completion"]
ref_texts  = ds_compl["completion"]

rouge_result = rouge.compute(
    predictions=pred_texts,
    references=ref_texts,
    use_aggregator=False,   # ← returns lists, one value per sample
)

print(rouge_result)  # {"rouge1": [...], "rouge2": [...], "rougeL": [...], "rougeLsum": [...]}